# 配置目标通过固定组成映射训练
3固定seed、各300轮、3个新原生账户；只修改配置损失接收的权重。原预测输出、平滑、控制器及最终一次组成映射保持。训练仍是代理，不声称完整训练部署一致。独立设计审查后执行。

In [1]:
from pathlib import Path
import os,json,qlib
from qlib.data import D
from qlib.workflow import R
root=Path.cwd()
while not (root/'SPEC.md').exists():root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';os.environ['MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR']='false'
card=root/'research/composition_objective_training_20261015';plan=json.loads((card/'plan.json').read_text())
qlib.init(provider_uri=str(root/plan['provider_uri']),region='us',kernels=1)
from src.workflow.composition_objective_training import run
summary=run(root=root,card=card,plan=plan)
print(json.dumps(summary,ensure_ascii=False,indent=2))

2026/10/15 14:59:40 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[49620:MainThread](2026-10-15 14:59:40,927) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[49620:MainThread](2026-10-15 14:59:40,928) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[49620:MainThread](2026-10-15 14:59:40,928) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "experiment": "composition_objective_training_20261015",
  "plan_sha256": "727edb8b7d997176ffe2dd29df1803383e3f80c9ac9b68e6cf80a7fe6acc9590",
  "decision": "not_promoted_fixed_composition_objective",
  "objective_achieved": false,
  "new_fits": 3,
  "new_native_cases": 3,
  "gates": {
    "beat_paired_all_costs": false,
    "beat_strong_always_all_costs": false,
    "net_soxx_excess_all_costs": false,
    "drawdown_30pct": false,
    "cvar_no_worse": true,
    "rolling_70pct": false,
    "complete_annual_70pct": false,
    "bootstrap_positive": false,
    "independent_development": false,
    "qualified_forward": false
  },
  "provider_uri": "/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013",
  "region": "us",
  "as_of": "2026-10-09",
  "recorder_id": "6f10179415094652b7107bcc506d7d50",
  "experiment_id": "517100394300994485",
  "recorder_uri": "/Users/rziiiii/dev/workspace/research/composition_objective_training_20261015/qlib_recorder",
  "recorder_local

In [2]:
# Post-result descriptive readback only: no fitting, policy simulation, or Recorder write.
import hashlib
import numpy as np
import pandas as pd
from src.workflow.minute_action_value import _source
from src.model.nominal_beta_composition import cancel_leverage_cash
layer_source = {'card': 'research/composition_objective_training_20261015', 'recorder_id': '6f10179415094652b7107bcc506d7d50', 'archive': '163d33490f075ca55273b0271aa88d370bb7707d9e6977f508b8cb8f946fef6a', 'inventory_sha256': 'b85c90edd9db21f00b0ddb8b92cb2f31f9bf2fc258a4a8e4e7b29e0d17ee3db6'}
saved, source_audit = _source(root, layer_source, [
    'analysis/predictions', 'analysis/targets', 'analysis/native/candidate_2',
    'analysis/accounting_attribution'])
dates = pd.DatetimeIndex(saved['analysis/native/candidate_2']['authority'].signal_date)
weight_columns = ['soxl_weight', 'soxx_weight', 'cash_weight']
rows = []
def add_weights(layer, family, seed, values):
    values = values.reindex(dates)
    assert len(values) == 944 and values[weight_columns].notna().all().all()
    rows.append(dict(layer=layer, family=family, seed=str(seed), slippage_bps=None,
        sessions=len(values), mean_soxl_pct=100*values.soxl_weight.mean(),
        mean_soxx_pct=100*values.soxx_weight.mean(), mean_cash_pct=100*values.cash_weight.mean(),
        mean_nominal_beta=(3*values.soxl_weight+values.soxx_weight).mean()))
for (family, seed), values in saved['analysis/predictions'].items():
    add_weights('raw_seed', family, seed, values)
for family in ['control', 'candidate']:
    values = sum(saved['analysis/predictions'][(family, seed)][weight_columns]
                 for seed in plan['seeds']) / len(plan['seeds'])
    add_weights('raw_equal_ensemble', family, 'all3', values)
    values = saved['analysis/targets'][family].reindex(dates)
    add_weights('smoothed_ensemble', family, 'all3', values)
    mapped = pd.DataFrame(np.array([cancel_leverage_cash(w) for w in values[weight_columns].to_numpy()]),
                          index=dates, columns=weight_columns)
    add_weights('T_smoothed_without_controller', family, 'all3', mapped)
actual = saved['analysis/accounting_attribution']
for row in actual[actual.family.isin(['candidate', 'posthoc_composition'])].itertuples():
    rows.append(dict(layer='actual_after_open', family='control' if row.family=='posthoc_composition' else 'candidate',
        seed='all3', slippage_bps=row.slippage_bps, sessions=row.sessions,
        mean_soxl_pct=None, mean_soxx_pct=None, mean_cash_pct=row.mean_cash_after_open_pct,
        mean_nominal_beta=None))
layer_stats = pd.DataFrame(rows)
layer_stats.to_csv(card/'layer_stats.csv', index=False)
metadata = dict(purpose='Post-result descriptive pipeline readback; not an executable no-controller counterfactual or causal effect.',
    source=layer_source, source_audit=source_audit, date_basis='Same944 native signal dates for every seed/layer.',
    control_definition='Frozen tangent model in raw/smoothed layers; same model with fixed posthoc composition for actual after-open cash.',
    missing_fields='Actual after-open asset weights/beta not inferred from cash-only accounting summary; intentionally null.',
    new_fits=0, new_native_accounts=0,
    sidecar_sha256=hashlib.sha256((card/'layer_stats.csv').read_bytes()).hexdigest())
(card/'layer_stats_source.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2)+'\n')
display(layer_stats)


,layer,family,seed,slippage_bps,sessions,mean_soxl_pct,mean_soxx_pct,mean_cash_pct,mean_nominal_beta
0,raw_seed,control,71,NaN,944,24.553845,43.087065,32.359091,1.167486
1,raw_seed,candidate,71,NaN,944,26.198968,32.080860,41.720171,1.106778
2,raw_seed,control,172,NaN,944,25.843665,40.123824,34.032511,1.176548
3,raw_seed,candidate,172,NaN,944,28.138721,27.392960,44.468319,1.118091
4,raw_seed,control,273,NaN,944,25.774365,41.044681,33.180953,1.183678
5,raw_seed,candidate,273,NaN,944,26.402582,32.441278,41.156140,1.116490
6,raw_equal_ensemble,control,all3,NaN,944,25.390625,41.418523,33.190852,1.175904
7,smoothed_ensemble,control,all3,NaN,944,25.385437,41.395194,33.219369,1.175515
8,T_smoothed_without_controller,control,all3,NaN,944,10.591116,85.778157,3.630728,1.175515
9,raw_equal_ensemble,candidate,all3,NaN,944,26.913424,30.638366,42.448210,1.113786


# 配置损失通过固定组成映射训练

完成3个固定seed（71/172/273）各300轮真实拟合，以及3个独立0/2/5bp完整Qlib原生账户。模型仍为39输入小配置头；每模型成熟主训练3014行、辅助20741行。初始化权重逐值匹配，训练域、成熟截止、scaler与冻结源一致，旧模型预测和平滑目标读回一致。三个seed全部参与等权集成，未挑选checkpoint或seed。

唯一模型干预：已有allocation_objective接收T后的权重，T将同时持有的SOXL与现金按固定名义beta转换为SOXX。所有配置损失项使用同一转换后权重；分类标签、系数、结构及梯度连接保持原定义，但共享编码器受到新配置梯度后，最终分类预测可以改变。推理仍输出原投影权重，经过原平滑和自身账户风控后，实际开盘/恢复目标只转换一次。

|滑点|训练候选净收益|冻结后处理组成对照|每天执行强参考|候选监控回撤|126日正超额比例|
|---|---:|---:|---:|---:|---:|
|0bp|60.64%|417.88%|358.11%|33.45%|9.16%|
|2bp|67.81%|435.28%|359.33%|31.68%|12.21%|
|5bp|53.83%|424.41%|351.07%|32.16%|16.24%|

SOXX同窗约387.7%。候选明显落后两个冻结对照与SOXX，所有费用账户监控回撤超过30%；滚动、年度、置信门失败，仅CVaR门通过。不存在可选出的合格成本情景。

2bp按trade_date分组，2023/2024/2025/2026相对SOXX收益分别-27.15%、-17.26%、-28.75%、-19.88%。2023与2026是截断年度，完整2024/2025也均失败。全部季度、全部819个连续126日窗口及状态归因保留，未删困难区间。2bp相对SOXX对数财富-1.06696，其中现金两项合计-0.99022、SOXL相对项-0.06181，直接费用不是主要缺口；这些是会计贡献，不是可无风险收回的收益。

## 训练代理与解释边界

实际冻结模型cash_target_penalty、absolute/conditional_drawdown、rolling5d及trade_event惩罚系数均为0，不是新增现金约束将候选推回现金。有效配置项仍为年度相对对数收益均值、0.5年度标准差、0.006换手项及10bp训练费用代理；分类系数0.02、辅助0.05，lr0.002、weight_decay0.08。原minimum_cash_weight=.15仍用于raw投影，inactive现金监督目标也保留原参数，未改变其系数。

T在训练时位于控制器之前，与部署端平滑/风险投影不交换；本次不是完整端到端执行训练。没有对原生费用、整数股、状态机或时钟反向传播。PyTorch2.11.0 minimum相等点固定采用0.5/0.5子梯度，不搜索平滑参数。训练代理表现不能替代完整账户。

三成本恢复分类器合格调用564/562/637次，恢复动作均0；没有降低阈值。候选目标名义beta转换误差最大1.67e-16，不代表真实风险相等。新完整账户的费用、现金、数量与原生Qlib核验通过；价格及原生报告时戳与冻结强参考相同。

## 决定与下一信息需求

not_promoted_fixed_composition_objective。该固定训练通过映射路线失败并关闭，不扫损失系数、风险惩罚、seed、epoch、阈值、窗口或映射比例挽救；先前更强的后处理组成参考保留原风险失败资格。总Goal仍未完成。

完成后的只读描述核对覆盖相同944日和全部三个seed：平滑集成原名义beta由1.1755降至1.1149，原现金由33.22%增至42.40%。但对平滑目标直接应用固定T、不接入控制器时，平均SOXL/SOXX/cash从10.59%/85.78%/3.63%变为6.93%/90.70%/2.37%；真实2bp开盘成交后现金却从22.58%增至45.17%。三个seed原beta均下降约0.06，没有挑单seed。

因此不能把失败简化为“转换后模型直接预测大量现金”：转换后的无控制器目标平均现金反而更低。大现金与实际状态路径共同出现；上述算术描述不是隔离控制器的因果反事实，也未执行去控制器账户。下一具体研究问题是训练代理遗漏的账户风险状态如何改变动作价值；必须先与已有端到端/状态训练失败路线去重，再登记不同机制，不能继续改本轮映射或旧风险阈值。

验证：tests/test_tangent_allocator.py共2项通过，包含转换后实际损失/梯度与直接公式一致、梯度与旧目标不同、推理方法继承不变及固定min子梯度。notebook执行完成；源初始化、scaler、成熟样本、预测和平滑目标在工作流中逐值核验。run_state保留恰好3fit尝试/3完成、3native尝试/3完成，无重复拟合或账户。

大模型、预测、目标、原生账户、日归因和完整滚动窗口仅在Recorder。瘦卡保留training_audit.csv、backtest_benchmark_comparison.csv、annual_metrics.csv、quarterly_attribution.csv、rolling_summary.csv、uncertainty.csv、composition_audit.csv及相应归因表。

Recorder：`6f10179415094652b7107bcc506d7d50`；冻结计划SHA `727edb8b7d997176ffe2dd29df1803383e3f80c9ac9b68e6cf80a7fe6acc9590`。

上述事后分层数字由notebook新增只读单元复现，见[layer_stats.csv](layer_stats.csv)及[layer_stats_source.json](layer_stats_source.json)。该单元固定原Recorder归档/清单身份，不写入原运行、不重新拟合或重放账户；actual_after_open只报告已核验的现金字段，未从现金推造股权或beta。
